# 02 - Capa Silver\n\nLimpieza, tipificación y normalización de datos.\n\nEn esta capa se transforma la fecha original `fecha_transaccion`, que viene con `/`, creando `fecha_venta` en formato estándar `yyyy-MM-dd`.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab usando Google Drive personal.

Estructura esperada en Drive:

```text
Proyecto_BigData_Forus/
├── raw/
│   └── venta_tiendas.csv
├── bronze/
├── silver/
├── gold/
└── evidencias/
```


In [1]:
import sys
import os

# Attempt to uninstall dataproc-spark-connect if it exists and causes conflicts
!pip uninstall -y dataproc-spark-connect || true

# Clear pip cache to ensure fresh installations
!pip cache purge

# Uninstall existing versions of pyspark and delta-spark
!pip uninstall -y pyspark delta-spark

# Install the specified versions
# We check for Python 3.12 compatibility as it often causes issues
if sys.version_info.major == 3 and sys.version_info.minor >= 12:
    print(f"Detected Python {sys.version_info.major}.{sys.version_info.minor}. Trying pyspark==3.5.3 and delta-spark==3.2.1 for compatibility.")
    !pip install pyspark==3.5.3 delta-spark==3.2.1
else:
    print(f"Detected Python {sys.version_info.major}.{sys.version_info.minor}. Installing pyspark==3.5.1 and delta-spark==3.2.0.")
    !pip install pyspark==3.5.1 delta-spark==3.2.0

# Verify installed versions
print("\n--- Installed PySpark and Delta-Spark versions ---")
!pip list | grep -E 'pyspark|delta-spark'
print("--------------------------------------------------\n")

# Workaround for the ImportError, try to force reload modules if they exist
# This might not be strictly necessary after a runtime restart but can help in some cases
if 'pyspark' in sys.modules:
    del sys.modules['pyspark']
if 'delta' in sys.modules:
    del sys.modules['delta']

Files removed: 11
Found existing installation: pyspark 3.5.3
Uninstalling pyspark-3.5.3:
  Successfully uninstalled pyspark-3.5.3
Found existing installation: delta-spark 3.2.1
Uninstalling delta-spark-3.2.1:
  Successfully uninstalled delta-spark-3.2.1
Detected Python 3.12. Trying pyspark==3.5.3 and delta-spark==3.2.1 for compatibility.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840629 sha256=91f556a41398b6e9c738708113e8673a269679e7c63ee1106b1ed883d614a269
  Stored in directory: /root/.cache/pip/wheels/07/a0/a3/d24c94bf043ab5c7e38c30491199a2a11fef8d2584e6df7fb7
Successfully built pyspark

--- Installed PySpark and Delta-Spark versions ---
delta-spark                              3.2.1
pyspark                                  3.5.3
--------------------------------------------------



In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

RUTA_BASE = "/content/drive/MyDrive/Proyecto_BigData_Forus"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"
RUTA_EVIDENCIAS = f"{RUTA_BASE}/evidencias"

ARCHIVO_VENTAS = "venta_tiendas.csv"

for ruta in [RUTA_RAW, RUTA_BRONZE, RUTA_SILVER, RUTA_GOLD, RUTA_EVIDENCIAS]:
    os.makedirs(ruta, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Ruta base:", RUTA_BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark: 3.5.3
Ruta base: /content/drive/MyDrive/Proyecto_BigData_Forus


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

RUTA_BASE = "/content/drive/MyDrive/Proyecto_BigData_Forus"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"
RUTA_EVIDENCIAS = f"{RUTA_BASE}/evidencias"

ARCHIVO_VENTAS = "venta_tiendas.csv"

for ruta in [RUTA_RAW, RUTA_BRONZE, RUTA_SILVER, RUTA_GOLD, RUTA_EVIDENCIAS]:
    os.makedirs(ruta, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Ruta base:", RUTA_BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark: 3.5.3
Ruta base: /content/drive/MyDrive/Proyecto_BigData_Forus


## 1. Lectura desde Bronze

In [9]:
import shutil
import os

source_file_path = f"/content/drive/MyDrive/{ARCHIVO_VENTAS}"
destination_folder_path = RUTA_RAW
destination_file_path = f"{RUTA_RAW}/{ARCHIVO_VENTAS}"

print(f"Attempting to move file from: {source_file_path}")
print(f"To folder: {destination_folder_path}")

if os.path.exists(source_file_path):
    try:
        # Ensure the destination directory exists
        os.makedirs(destination_folder_path, exist_ok=True)
        shutil.move(source_file_path, destination_file_path)
        print(f"Successfully moved '{ARCHIVO_VENTAS}' to '{destination_folder_path}'")
        print("Please re-run the file existence check (cell 852125a2) and then the data processing cell (8b1badf8).")
    except Exception as e:
        print(f"Error moving file: {e}")
        print("It's recommended to move the file manually via the Google Drive web interface if this fails.")
else:
    print(f"Error: Source file '{source_file_path}' does not exist. No file to move.")
    print("Please ensure 'venta_tiendas.csv' is in the root of your Google Drive or move it there manually.")

Attempting to move file from: /content/drive/MyDrive/venta_tiendas.csv
To folder: /content/drive/MyDrive/Proyecto_BigData_Forus/raw
Successfully moved 'venta_tiendas.csv' to '/content/drive/MyDrive/Proyecto_BigData_Forus/raw'
Please re-run the file existence check (cell 852125a2) and then the data processing cell (8b1badf8).


In [10]:
import os

# Define paths
delta_path = f"{RUTA_BRONZE}/venta_tiendas"
raw_csv_path = f"{RUTA_RAW}/{ARCHIVO_VENTAS}"

# Check if the Delta table already exists. If not, create it from raw CSV.
# This ensures that subsequent reads of the bronze layer have data.
# We check for the existence of the _delta_log directory as a robust indicator of a Delta table.
if not os.path.exists(delta_path) or not os.path.exists(f"{delta_path}/_delta_log"):
    print(f"Delta table not found at {delta_path}. Creating from raw CSV: {raw_csv_path}")

    # Read the raw CSV file
    df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_csv_path)

    # Write to Delta table in bronze layer
    df_raw.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Delta table created at {delta_path}")
else:
    print(f"Delta table already exists at {delta_path}. Skipping creation from raw CSV.")

# Now, read the Delta table from the bronze layer
df_bronze = spark.read.format("delta").load(delta_path)

print("Registros Bronze:", df_bronze.count())
df_bronze.printSchema()
df_bronze.show(5, truncate=False)


Delta table not found at /content/drive/MyDrive/Proyecto_BigData_Forus/bronze/venta_tiendas. Creating from raw CSV: /content/drive/MyDrive/Proyecto_BigData_Forus/raw/venta_tiendas.csv
Delta table created at /content/drive/MyDrive/Proyecto_BigData_Forus/bronze/venta_tiendas
Registros Bronze: 2250970
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: integer (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: integer (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: integer (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: integer (nullable = true)
 |-- costo: integer (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_bol

In [7]:
import os

file_path_to_check = f"{RUTA_RAW}/{ARCHIVO_VENTAS}"

if os.path.exists(file_path_to_check):
    print(f"The file '{ARCHIVO_VENTAS}' exists at: {file_path_to_check}")
else:
    print(f"ERROR: The file '{ARCHIVO_VENTAS}' does NOT exist at: {file_path_to_check}")
    print("Please ensure the file is uploaded to this exact location in your Google Drive.")

ERROR: The file 'venta_tiendas.csv' does NOT exist at: /content/drive/MyDrive/Proyecto_BigData_Forus/raw/venta_tiendas.csv
Please ensure the file is uploaded to this exact location in your Google Drive.


## 2. Transformaciones Silver\n\nReglas aplicadas:\n\n- Conversión de columnas numéricas.\n- Normalización de `fecha_transaccion`.\n- Creación de `fecha_venta` como tipo `date`.\n- Creación de `fecha_venta_texto` con guiones.\n- Cálculo de `margen` y `margen_porcentaje`.\n- Eliminación de registros inválidos críticos.

In [11]:
from pyspark.sql.functions import (
    col, regexp_replace, trim, to_timestamp, to_date, date_format,
    year, month, dayofmonth, when, lit
)

inicio = time.time()

# Limpieza del sufijo " CL" y parseo de fecha original
fecha_sin_zona = regexp_replace(col("fecha_transaccion"), " CL$", "")

df_silver = (
    df_bronze
    .withColumn("id_canal", col("id_canal").cast("int"))
    .withColumn("numero_transaccion", col("numero_transaccion").cast("long"))
    .withColumn("numero_pos", col("numero_pos").cast("int"))
    .withColumn("numero_boleta", col("numero_boleta").cast("long"))
    .withColumn("cod_tienda_facturacion", col("cod_tienda_facturacion").cast("int"))
    .withColumn("id_producto", col("id_producto").cast("long"))
    .withColumn("unidades", col("unidades").cast("int"))
    .withColumn("venta", col("venta").cast("double"))
    .withColumn("costo", col("costo").cast("double"))
    .withColumn("tipo_documento", trim(col("tipo_documento")))

    # Transformación solicitada: slash a guion
    .withColumn("fecha_transaccion_guion", regexp_replace(col("fecha_transaccion"), "/", "-"))

    # Conversión profesional a timestamp/date
    .withColumn("fecha_timestamp", to_timestamp(fecha_sin_zona, "dd/MM/yyyy hh:mm:ss a"))
    .withColumn("fecha_venta", to_date(col("fecha_timestamp")))
    .withColumn("fecha_venta_texto", date_format(col("fecha_venta"), "yyyy-MM-dd"))

    # Variables derivadas comerciales
    .withColumn("anio", year(col("fecha_venta")))
    .withColumn("mes", month(col("fecha_venta")))
    .withColumn("dia", dayofmonth(col("fecha_venta")))
    .withColumn("margen", col("venta") - col("costo"))
    .withColumn(
        "margen_porcentaje",
        when(col("venta") > 0, (col("venta") - col("costo")) / col("venta")).otherwise(lit(None))
    )
)

# Reglas mínimas de calidad
df_silver = (
    df_silver
    .filter(col("fecha_venta").isNotNull())
    .filter(col("id_producto").isNotNull())
    .filter(col("venta").isNotNull())
    .filter(col("unidades").isNotNull())
)

tiempo_transformacion = time.time() - inicio

print("Registros Silver:", df_silver.count())
print("Tiempo transformación Silver:", round(tiempo_transformacion, 2), "segundos")
df_silver.printSchema()
df_silver.select("fecha_transaccion", "fecha_transaccion_guion", "fecha_venta", "fecha_venta_texto").show(10, truncate=False)


Registros Silver: 2250970
Tiempo transformación Silver: 1.03 segundos
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: long (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: long (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: long (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: double (nullable = true)
 |-- costo: double (nullable = true)
 |-- fecha_transaccion_guion: string (nullable = true)
 |-- fecha_timestamp: timestamp (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- fecha_venta_texto: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- margen: double (nullable = true)
 |-- margen_porcentaje: double (nullable = true)

+-------------------------+-------------------------

In [15]:
# ============================================================
# Validación de calidad de fecha
# ============================================================

registros_fecha_nula = df_silver.filter(
    col("fecha_venta").isNull()
).count()

print("Registros con fecha_venta nula o inválida:", registros_fecha_nula)

if registros_fecha_nula > 0:
    print("Advertencia: existen registros con fecha inválida que deben revisarse.")
else:
    print("Validación correcta: todas las fechas fueron convertidas exitosamente.")

Registros con fecha_venta nula o inválida: 0
Validación correcta: todas las fechas fueron convertidas exitosamente.


## 3. Validaciones de calidad

In [12]:
from pyspark.sql.functions import sum as spark_sum

validaciones = df_silver.select(
    spark_sum(when(col("fecha_venta").isNull(), 1).otherwise(0)).alias("fechas_invalidas"),
    spark_sum(when(col("id_producto").isNull(), 1).otherwise(0)).alias("productos_invalidos"),
    spark_sum(when(col("venta") < 0, 1).otherwise(0)).alias("ventas_negativas"),
    spark_sum(when(col("unidades") <= 0, 1).otherwise(0)).alias("unidades_no_positivas")
)

validaciones.show()


+----------------+-------------------+----------------+---------------------+
|fechas_invalidas|productos_invalidos|ventas_negativas|unidades_no_positivas|
+----------------+-------------------+----------------+---------------------+
|               0|                  0|          197119|               197008|
+----------------+-------------------+----------------+---------------------+



## 4. Escritura Silver particionada\n\nSe particiona por `anio` y `mes` para mejorar filtros temporales y consultas analíticas.

In [13]:
inicio = time.time()

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("anio", "mes")
    .save(f"{RUTA_SILVER}/venta_tiendas")
)

tiempo_silver = time.time() - inicio

print("Silver escrito correctamente en:", f"{RUTA_SILVER}/venta_tiendas")
print("Tiempo escritura Silver:", round(tiempo_silver, 2), "segundos")


Silver escrito correctamente en: /content/drive/MyDrive/Proyecto_BigData_Forus/silver/venta_tiendas
Tiempo escritura Silver: 63.35 segundos


## 5. Linaje Silver

In [14]:
linaje_silver = {
    "dataset": "venta_tiendas",
    "origen": f"{RUTA_BRONZE}/venta_tiendas",
    "destino": f"{RUTA_SILVER}/venta_tiendas",
    "capa": "Silver",
    "formato": "Delta Lake",
    "transformaciones": [
        "Casting de variables numéricas",
        "fecha_transaccion con slash transformada a fecha_transaccion_guion",
        "Creación de fecha_venta tipo date en formato yyyy-MM-dd",
        "Cálculo de margen y margen_porcentaje",
        "Particionamiento por anio y mes",
        "Filtros de calidad sobre fecha, producto, venta y unidades"
    ]
}

for k, v in linaje_silver.items():
    print(f"{k}: {v}")


dataset: venta_tiendas
origen: /content/drive/MyDrive/Proyecto_BigData_Forus/bronze/venta_tiendas
destino: /content/drive/MyDrive/Proyecto_BigData_Forus/silver/venta_tiendas
capa: Silver
formato: Delta Lake
transformaciones: ['Casting de variables numéricas', 'fecha_transaccion con slash transformada a fecha_transaccion_guion', 'Creación de fecha_venta tipo date en formato yyyy-MM-dd', 'Cálculo de margen y margen_porcentaje', 'Particionamiento por anio y mes', 'Filtros de calidad sobre fecha, producto, venta y unidades']
